In [1]:
import ollama
import numpy as np
from IPython.display import display, Markdown
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os
import json
from pypdf import PdfReader

/home/growlt366/Public/Projects/Private Knowledge assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## RAG Pipeline Phase-1 Document Uploading

1. Document loading 
2. Chunking                               (Todo : Finding scalable chunking strategy)
3. Generating Chunk vector embedding 
4. Storing Vector Embedding 

## RAG Pipeline Phase-2 Quering Document 
1. Query input from user
2. Classify the query                      (Todo)
3. Converting query into embedding 
4. Finding relavant top_3 chunk            (Todo : finding scalable chunking and retrieval strategy for large documents)
5. Extracting chunk into context
6. LLM call to refactor context in relevant manner.   (Todo)
7. Generating Prompt 
8. LLM call.
9. Final output.

# Phase : 1

In [204]:
with open("sample.txt", "r") as file:
    text = file.read()

# chunking 
chunks = text.split(".")
new_chunk = []

for i in range(len(chunks)):
    temp = chunks[i].replace('\n',' ')
    if len(temp.strip())>0:
        new_chunk.append(temp)

chunks = new_chunk 


new_chunk = []
window_size = 3
for i in range(len(chunks)-window_size+1):
    new_chunk.append("".join([chunks[i+j] for j in range(window_size)]))
sentence_chunks = chunks 
chunks = new_chunk

In [205]:
reader = PdfReader('knowledge.pdf')

data = ""
chunks = []

for page in reader.pages:
    data = data + page.extract_text()

words_in_data = len(data)
for i in range(0,words_in_data,400):
    chunks.append(data[i:i+500])
print(f"we have generated {len(chunks)} chunks")

we have generated 362 chunks


In [198]:
len(chunks[46])

900

In [207]:
#embedding
vector_list = []
for i in range(len(chunks)):
    vector_list.append(ollama.embeddings(prompt = chunks[i], model="mxbai-embed-large"))
vector_db = np.array([np.array(i['embedding']) for i in vector_list])

# Phase 2 :

In [211]:
def cosine_similarity(a,b):
    dot_product = np.dot(a,b) 
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

In [212]:
def get_embedding(prompt):
    return np.array(ollama.embeddings(prompt = prompt, model="mxbai-embed-large")['embedding'])

In [213]:
def get_relevant_chunk_index(query_embedding , k = len(vector_db)):
    similarity_vector = []
    top_k = []
    for i in vector_db:
        similarity_vector.append(cosine_similarity(query_embedding,i))
    top_k =  [index for index ,value in sorted(list(enumerate(similarity_vector)),key= lambda x : x[1],reverse= True)[:k]]
    return top_k
    

In [2]:
def LLM_call(prompt,model):
    return ollama.generate(
        model = model,
        prompt = prompt
    )['response']


In [73]:
# Query classification LLM call
# document specific example (todo) ------- pending
def classify_query(query):

    # Important and Mandatory : Output only the category name.
    classify_prompt = """
        ### Role
        You are a precision linguistic analyst specializing in query intent classification.

        ### Task
        Classify the user's input <query> into exactly one of the following five classes.

        ### constraints 
        - strictly follow the output format : {{class : string , reason : string}}
        - don't generate any extra content other than as specified by output format.
        - no need of explanation.
        - don't generate any precedding explanation before the actual required reponse as described by output format.

        ### Definitions
        1. factual: Questions seeking a direct lookup, specific fact, or definition (e.g., "What is the capital of France?").
        2. reasoning: Questions asking "why" or "how," requiring an explanation of a process or cause (e.g., "How does photosynthesis work?").
        3. comparison: Queries seeking differences, similarities, or a side-by-side evaluation of two or more entities (e.g., "Difference between SQL and NoSQL").
        4. multi-hop: Complex questions where multiple concepts must be linked or several steps are needed to reach the answer (e.g., "Who was the president when the person who invented the telephone died?").
        5. open-ended: Vague, broad, or subjective prompts that lack a specific answer or clear constraints (e.g., "Tell me about life").

        ### Guidelines
        - Think step-by-step: First, analyze the structure and intent of the query.
        - Then, select the most appropriate class.
        - Output Format:
          {{ class : string, reason : string}}

        ### Examples
        Input: "Why do leaves change color in autumn?"
        Output: {{"class": "reasoning", "reason: "The query asks for the 'why' behind a natural process, requiring an explanation of cause and effect."}}

        Input: "Compare the iPhone 15 and Samsung S24."
        Output: {{"class": "comparison", "reason": "The query explicitly asks for a comparison between two specific products.",}}

        ### Input to Classify
        <query>
        {user_query}
        </query>

        
        ### Output 
        {{
            class : string,
            reason : string
        }}
    """
    
    prompt = classify_prompt.format(user_query=query)
    # print(prompt)

    return LLM_call(prompt,"llama3.2")

In [74]:
queries = ["What is the boiling point of nitrogen?",
           "Define 'quantum entanglement' in simple terms.",
           "Why do cats purr even when they are in pain?",
           "How does a blockchain achieve consensus?",
           "Laptops vs Tablets for college students",
           "What's the difference between a virus and a bacterium?",
           "Who directed the movie that won Best Picture the year I was born (1995)?",
           "Find the net worth of the person who founded the company that created Java.",
           "Should I move to a new city?",
           "Thoughts on the future of humanity?"]
response = []
for query in queries :
    response.append(classify_query(query))

In [76]:
response

[' {"class": "factual", "reason": "The query seeks a direct lookup of a specific fact, namely the boiling point of nitrogen."}',
 ' {"class": "factual", "reason": "The query seeks a definition of a specific term, indicating it is looking for a direct lookup or specific fact."}',
 '{"class": "reasoning", "reason": "The query asks for the \'why\' behind a biological phenomenon, requiring an explanation of cause and effect."}',
 '{"class": "reasoning", "reason": "The query asks for the \'why\' behind a process, requiring an explanation of cause and effect."}',
 ' {"class": "comparison", "reason": "The query explicitly asks for a comparison between two specific entities, in this case, laptops and tablets."}',
 ' {"class": "comparison", "reason": "The query explicitly asks for a comparison between two specific types of microorganisms."}',
 '{"class": "multi-hop", "reason": "The query requires linking multiple concepts: the year of birth (1995), the Best Picture winner, and the movie\'s dire

In [75]:
# json.loads(response[0])['reason']
for i in range(len(response)) :
    classification = json.loads(response[i])
    output_class = classification['class']
    classification_reason = classification['reason']
    print(f"{i+1}.  class : {output_class}, reason : {classification_reason}")

1.  class : factual, reason : The query seeks a direct lookup of a specific fact, namely the boiling point of nitrogen.
2.  class : factual, reason : The query seeks a definition of a specific term, indicating it is looking for a direct lookup or specific fact.
3.  class : reasoning, reason : The query asks for the 'why' behind a biological phenomenon, requiring an explanation of cause and effect.
4.  class : reasoning, reason : The query asks for the 'why' behind a process, requiring an explanation of cause and effect.
5.  class : comparison, reason : The query explicitly asks for a comparison between two specific entities, in this case, laptops and tablets.
6.  class : comparison, reason : The query explicitly asks for a comparison between two specific types of microorganisms.
7.  class : multi-hop, reason : The query requires linking multiple concepts: the year of birth (1995), the Best Picture winner, and the movie's director.
8.  class : multi-hop, reason : The query requires find

In [ ]:
def handle_query_context(user_query,query_class,query_embedding):
    # factual -> extract top 3 chunks 
    # reasoning -> extract top 5 chunks
    # comparision -> query decomposition 
    # multi hop -> query decomposition and extracting for each query and merging the context
    # open-ended -> extracting more chunks for elaborated resposne 

    pass

In [ ]:
# Response Generation LLM call
prompt_template = """
        You are an helpful assistant. 
        You are good with framing human like informative and short answers based on given context.
        Generate response after thorough understanding of the context in simple and polite way.
        Use the technical wording as per context as and when required as per question.
        If the answer is not present in context just repond "I don't know".

        context : {context}

        question : {user_query}
"""
def rag(user_query):
    query_embedding = get_embedding(user_query)
    query_class = classify_query(user_query)
    context = handle_query_context(user_query,query_class,query_embedding)
    top_k_chunk = get_relevant_chunk_index(query_embedding, window_size)

    unique_top_k_chunk = []
    for index in top_k_chunk :
        unique_top_k_chunk.extend([i for i in range(index,index+window_size)])
    unique_top_k_chunk  = set(unique_top_k_chunk)
    context = "".join([sentence_chunks[i] for i in unique_top_k_chunk])

    prompt = prompt_template.format(context = context, user_query = user_query)
    response = LLM_call(prompt = prompt ,model = "gemma4")

    return response,context

In [ ]:
questions = ['What is RAG?',
            'Why is chunk size important?',
            'What similarity measures are used?',
            'How can retrieval quality be improved?',
            'What is the difference between hybrid search and vector search?',
            'Does RAG completely eliminate hallucinations?',
            'What is reinforcement learning?']

responses = []
contexts = []

for i in questions:
    response, context = rag(i)
    responses.append(response)
    contexts.append(context)

In [59]:
for i in range(len(responses)):
    print(f"Ans {i+1} : ")
    display(Markdown(responses[i]))
    print(f"Context {i+1}")
    display(Markdown(contexts[i]))
    print("\n")

Ans 1 : 


Retrieval-Augmented Generation (RAG) is a framework designed to enhance language models by integrating external knowledge retrieval. Instead of relying solely on parametric memory, RAG fetches relevant information from a knowledge base before generating responses.

Context 1


Retrieval-Augmented Generation (RAG) is a framework that enhances language models by integrating external knowledge retrieval Instead of relying solely on parametric memory, RAG fetches relevant information from a knowledge base before generating responses  One of the core components of RAG is chunking Documents are divided into smaller segments before being embedded It reduces them by grounding responses in retrieved context  Efficient indexing and metadata tagging can further improve retrieval performance by narrowing down search space  RAG systems are widely used in applications such as chatbots, enterprise search, and document question answering systems



Ans 2 : 


...

  A major challenge in RAG systems is retrieval accuracy If irrelevant chunks are retrieved, the language model may generate incorrect or hallucinated answers  Another challenge is context window limitation  Hybrid search combines keyword-based and vector-based retrieval to improve performance, especially when exact matches are important  Despite its advantages, RAG does not completely eliminate hallucinations It reduces them by grounding responses in retrieved context  Efficient indexing and metadata tagging can further improve retrieval performance by narrowing down search space



Ans 7 : 


I don't know

Context 7


  A major challenge in RAG systems is retrieval accuracy If irrelevant chunks are retrieved, the language model may generate incorrect or hallucinated answers  Another challenge is context window limitation It involves re-evaluating retrieved candidates using additional signals  Query understanding is an emerging area in RAG systems Instead of treating all queries equally, systems can classify queries and adapt retrieval strategies accordingly  For example, factual queries require precise chunk retrieval, while analytical queries may require multiple chunks from different sections

In [62]:
responses

['Retrieval-Augmented Generation (RAG) is a framework designed to enhance language models by integrating external knowledge retrieval. Instead of relying solely on parametric memory, RAG fetches relevant information from a knowledge base before generating responses.',
 'Chunk size is crucial because it affects the preservation of meaning and context within the segmented documents. Specifically:\n\n*   **Large chunks** risk diluting the semantic meaning.\n*   **Small chunks** risk losing the necessary context.',
 'Similarity search is typically performed using **cosine similarity**, although other metrics, such as the **dot product**, can also be used.',
 'Retrieval quality can be improved through several methods. One key approach is using **re-ranking**, which is performed after initial retrieval to enhance the quality of selected chunks by re-evaluating candidates using additional signals. Furthermore, **query understanding** is cited as an emerging area within RAG systems that contri